In [15]:
# Imports and Setup
import pandas as pd
import numpy as np
import os
from datetime import datetime

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# Paths
SILVER_PATH = "../data/silver/"
GOLD_PATH = "../data/gold/"

# Create gold directory if it doesn't exist
if not os.path.exists(GOLD_PATH):
    os.makedirs(GOLD_PATH)
    print(f"Created directory: {GOLD_PATH}")

print("="*60)
print("GOLD LAYER NOTEBOOK - SETUP")
print("="*60)
print(f"Silver path: {SILVER_PATH}")
print(f"Gold path: {GOLD_PATH}")

GOLD LAYER NOTEBOOK - SETUP
Silver path: ../data/silver/
Gold path: ../data/gold/


In [16]:
# Load all silver layer files
print("\nLoading silver layer files...")
cert_silver = pd.read_csv(os.path.join(SILVER_PATH, "certifications_silver.csv"))
clubs_silver = pd.read_csv(os.path.join(SILVER_PATH, "clubs_silver.csv"))
results_silver = pd.read_csv(os.path.join(SILVER_PATH, "results_silver.csv"))

print(f"Certifications loaded: {cert_silver.shape}")
print(f"Clubs loaded: {clubs_silver.shape}")
print(f"Results loaded: {results_silver.shape}")


Loading silver layer files...
Certifications loaded: (21001, 16)
Clubs loaded: (436, 20)
Results loaded: (112296, 20)


In [ ]:
# CREATE DIM_ATHLETE (Athlete Master Data)
print("\n" + "="*60)
print("CREATING: dim_athlete")
print("="*60)

# Start from certifications (athlete master)
dim_athlete = cert_silver[['Code', 'person_type_clean', 'gender_clean', 'DOB', 'Age', 
                             'Mental Handicap (SOB has this certificate)', 
                             'Parents Consent (SOB has this certificate)',
                             'HAP (SOB has this certificate)',
                             'Unified Partner (SOB has this certificate)',
                             'dob_missing', 'person_type_missing']].copy()

# Rename columns for clarity
dim_athlete = dim_athlete.rename(columns={
    'Code': 'athlete_id',
    'person_type_clean': 'person_type',
    'gender_clean': 'gender',
    'DOB': 'date_of_birth',
    'Mental Handicap (SOB has this certificate)': 'mental_handicap_flag',
    'Parents Consent (SOB has this certificate)': 'parents_consent_flag',
    'HAP (SOB has this certificate)': 'hap_flag',
    'Unified Partner (SOB has this certificate)': 'unified_partner_flag'
})

# Remove duplicates 
dim_athlete = dim_athlete.drop_duplicates(subset=['athlete_id'], keep='first')

# Add data quality fields
dim_athlete['athlete_quality_flag'] = ((dim_athlete['dob_missing'] == 0) & 
                                        (dim_athlete['person_type_missing'] == 0)).astype(int)

# Add metadata
dim_athlete['gold_created_timestamp'] = datetime.now()

print(f"Shape: {dim_athlete.shape}")
print(f"Athletes with complete data: {dim_athlete['athlete_quality_flag'].sum()} ({dim_athlete['athlete_quality_flag'].sum()/len(dim_athlete)*100:.2f}%)")
print(f"Sample:\n{dim_athlete.head(3)}")

# Save to gold
output_path = os.path.join(GOLD_PATH, "dim_athlete.csv")
dim_athlete.to_csv(output_path, index=False)
print(f"\nSaved to: {output_path}")


CREATING: dim_athlete
Shape: (20222, 13)
Athletes with complete data: 19011 (94.01%)
Sample:
         athlete_id person_type gender date_of_birth   Age  \
0  3B1D6O9Z08W7IOZG     athlete      M    1980-04-28  45.0   
1  4PQUISSS7HAL3BVW       coach      M           NaN   0.0   
2  5YFHEPFDCRU2M1PA     athlete      M    1963-07-21  62.0   

   mental_handicap_flag  parents_consent_flag  hap_flag  unified_partner_flag  \
0                     1                     1         1                     0   
1                     0                     0         0                     0   
2                     1                     1         0                     0   

   dob_missing  person_type_missing  athlete_quality_flag  \
0            0                    0                     1   
1            1                    0                     0   
2            0                    0                     1   

      gold_created_timestamp  
0 2026-05-24 13:11:32.895746  
1 2026-05-24 13:11:32.895

In [5]:
# CREATE DIM_SPORT (Sport Master Data)
print("\n" + "="*60)
print("CREATING: dim_sport")
print("="*60)

# Extract unique sports from results
dim_sport = results_silver[['sport_clean']].drop_duplicates().copy()
dim_sport = dim_sport.dropna()

# Add sport_id
dim_sport = dim_sport.reset_index(drop=True)
dim_sport['sport_id'] = range(1, len(dim_sport) + 1)

# Rename and reorder columns
dim_sport = dim_sport.rename(columns={'sport_clean': 'sport_name'})
dim_sport = dim_sport[['sport_id', 'sport_name']]

# Add metadata
dim_sport['gold_created_timestamp'] = datetime.now()

print(f"Shape: {dim_sport.shape}")
print(f"Sports found: {len(dim_sport)}")
print(f"Sample:\n{dim_sport.head(10)}")

# Save to gold
output_path = os.path.join(GOLD_PATH, "dim_sport.csv")
dim_sport.to_csv(output_path, index=False)
print(f"\nSaved to: {output_path}")


CREATING: dim_sport
Shape: (23, 3)
Sports found: 23
Sample:
   sport_id             sport_name     gold_created_timestamp
0         1              Athletics 2026-05-24 10:38:01.670461
1         2                Cycling 2026-05-24 10:38:01.670461
2         3               Football 2026-05-24 10:38:01.670461
3         4             Sportgames 2026-05-24 10:38:01.670461
4         5  Gymnastics (Artistic) 2026-05-24 10:38:01.670461
5         6               Aquatics 2026-05-24 10:38:01.670461
6         7              Badminton 2026-05-24 10:38:01.670461
7         8           Table Tennis 2026-05-24 10:38:01.670461
8         9                  Bocce 2026-05-24 10:38:01.670461
9        10             Equestrian 2026-05-24 10:38:01.670461

Saved to: ../data/gold/dim_sport.csv


In [6]:
# CREATE DIM_REGION (Region Master Data)
print("\n" + "="*60)
print("CREATING: dim_region")
print("="*60)

dim_region = clubs_silver[['region']].drop_duplicates().copy()
dim_region = dim_region.dropna()
dim_region = dim_region.reset_index(drop=True)

# Add region_id
dim_region['region_id'] = range(1, len(dim_region) + 1)
dim_region = dim_region.rename(columns={'region': 'region_name'})
dim_region = dim_region[['region_id', 'region_name']]

# Add metadata
dim_region['gold_created_timestamp'] = datetime.now()

print(f"Shape: {dim_region.shape}")
print(f"Regions found: {len(dim_region)}")
print(f"Sample:\n{dim_region.head()}")

# Save to gold
output_path = os.path.join(GOLD_PATH, "dim_region.csv")
dim_region.to_csv(output_path, index=False)
print(f"\nSaved to: {output_path}")


CREATING: dim_region
Shape: (24, 3)
Regions found: 24
Sample:
   region_id        region_name     gold_created_timestamp
0          1            Hainaut 2026-05-24 10:38:03.751985
1          2         Luxembourg 2026-05-24 10:38:03.751985
2          3     Brabant Wallon 2026-05-24 10:38:03.751985
3          4              Liège 2026-05-24 10:38:03.751985
4          5  Brussel/Bruxelles 2026-05-24 10:38:03.751985

Saved to: ../data/gold/dim_region.csv


In [7]:
# CREATE DIM_CLUB (Club Master Data)
print("\n" + "="*60)
print("CREATING: dim_club")
print("="*60)

dim_club = clubs_silver[['club_id', 'club_name', 'region', 'total_participations']].copy()

# Join with region to get region_id
dim_club = dim_club.merge(dim_region, left_on='region', right_on='region_name', how='left')

# Count certified athletes per club
athlete_per_club = cert_silver[cert_silver['person_type_missing'] == 0].groupby('Club')['Code'].count().reset_index()
athlete_per_club = athlete_per_club.rename(columns={'Club': 'club_name', 'Code': 'certified_athlete_count'})

dim_club = dim_club.merge(athlete_per_club, on='club_name', how='left')
dim_club['certified_athlete_count'] = dim_club['certified_athlete_count'].fillna(0).astype(int)

# Select and reorder columns
dim_club = dim_club[['club_id', 'club_name', 'region_id', 'region_name', 'total_participations', 'certified_athlete_count']]

# Add metadata
dim_club['gold_created_timestamp'] = datetime.now()

print(f"Shape: {dim_club.shape}")
print(f"Total clubs: {len(dim_club)}")
print(f"Sample:\n{dim_club.head(5)}")

# Save to gold
output_path = os.path.join(GOLD_PATH, "dim_club.csv")
dim_club.to_csv(output_path, index=False)
print(f"\nSaved to: {output_path}")


CREATING: dim_club
Shape: (436, 7)
Total clubs: 436
Sample:
   club_id             club_name  region_id     region_name  \
0      100            LA PILERIE        1.0         Hainaut   
1      101                   BAM        2.0      Luxembourg   
2      102  LES DAHUTS - ANDAGE         2.0      Luxembourg   
3      103   LES TUNIQUES BLEUES        3.0  Brabant Wallon   
4      104               EVASION        1.0         Hainaut   

   total_participations  certified_athlete_count     gold_created_timestamp  
0                     9                      182 2026-05-24 10:38:05.852591  
1                     3                        6 2026-05-24 10:38:05.852591  
2                     4                       14 2026-05-24 10:38:05.852591  
3                     9                       45 2026-05-24 10:38:05.852591  
4                     6                       19 2026-05-24 10:38:05.852591  

Saved to: ../data/gold/dim_club.csv


In [8]:
# CREATE FACT_ATHLETE_RESULTS 
print("\n" + "="*60)
print("CREATING: fact_athlete_results")
print("="*60)

# Start from results
fact_results = results_silver[['Code', 'year', 'sport_clean', 'Club', 
                                'rank_numeric', 'score_numeric', 'is_disqualified']].copy()

# Rename columns
fact_results = fact_results.rename(columns={
    'Code': 'athlete_id',
    'year': 'competition_year',
    'sport_clean': 'sport_name',
    'Club': 'club_name',
    'rank_numeric': 'rank',
    'score_numeric': 'score'
})

# Join with dim_sport to get sport_id
fact_results = fact_results.merge(dim_sport[['sport_id', 'sport_name']], 
                                  on='sport_name', how='left')

# Join with dim_club to get club_id
fact_results = fact_results.merge(dim_club[['club_id', 'club_name']], 
                                  on='club_name', how='left')

# Add data quality flags
fact_results['score_missing_flag'] = fact_results['score'].isna().astype(int)
fact_results['rank_missing_flag'] = fact_results['rank'].isna().astype(int)

# Select and reorder columns
fact_results = fact_results[['athlete_id', 'competition_year', 'sport_id', 'sport_name',
                              'club_id', 'club_name', 'rank', 'score', 'is_disqualified',
                              'score_missing_flag', 'rank_missing_flag']]

# Add metadata
fact_results['gold_created_timestamp'] = datetime.now()

print(f"Shape: {fact_results.shape}")
print(f"Total results records: {len(fact_results)}")
print(f"Disqualifications: {fact_results['is_disqualified'].sum()} ({fact_results['is_disqualified'].sum()/len(fact_results)*100:.2f}%)")
print(f"Sample:\n{fact_results.head(5)}")

# Save to gold
output_path = os.path.join(GOLD_PATH, "fact_athlete_results.csv")
fact_results.to_csv(output_path, index=False)
print(f"\nSaved to: {output_path}")


CREATING: fact_athlete_results
Shape: (112296, 12)
Total results records: 112296
Disqualifications: 2241 (2.00%)
Sample:
         athlete_id  competition_year  sport_id sport_name  club_id club_name  \
0  001O91NNW62RZP97              2015         1  Athletics    550.0  DE LOVIE   
1  001O91NNW62RZP97              2015         1  Athletics    550.0  DE LOVIE   
2  001O91NNW62RZP97              2015         1  Athletics    550.0  DE LOVIE   
3  00BKM94J4INLRQAF              2015         2    Cycling    561.0  DE OEVER   
4  00BKM94J4INLRQAF              2015         2    Cycling    561.0  DE OEVER   

   rank  score  is_disqualified  score_missing_flag  rank_missing_flag  \
0   1.0   18.0                0                   0                  0   
1   2.0    1.0                0                   0                  0   
2   2.0    0.0                0                   0                  0   
3   4.0   13.0                0                   0                  0   
4   NaN    5.0       

In [ ]:
# CREATE FACT_ATHLETE_PARTICIPATION (Athlete-Club-Year Participation)
print("\n" + "="*60)
print("CREATING: fact_athlete_participation")
print("="*60)

# Build participation facts from results (athlete + club + year combinations)
fact_participation = results_silver.groupby(['Code', 'year', 'Club']).agg({
    'sport_clean': 'count',  
    'rank_numeric': 'count'  
}).reset_index()

fact_participation = fact_participation.rename(columns={
    'Code': 'athlete_id',
    'year': 'participation_year',
    'Club': 'club_name',
    'sport_clean': 'competition_count',
    'rank_numeric': 'valid_rank_count'
})

# Count unique sports per athlete-year
sports_per_athlete_year = results_silver.groupby(['Code', 'year'])['sport_clean'].nunique().reset_index()
sports_per_athlete_year = sports_per_athlete_year.rename(columns={
    'Code': 'athlete_id',
    'year': 'participation_year',
    'sport_clean': 'unique_sport_count'
})

fact_participation = fact_participation.merge(sports_per_athlete_year, 
                                              on=['athlete_id', 'participation_year'], 
                                              how='left')

# Flag multi-sport participation
fact_participation['multi_sport_flag'] = (fact_participation['unique_sport_count'] > 1).astype(int)

# Join with dim_club to get club_id
fact_participation = fact_participation.merge(dim_club[['club_id', 'club_name']], 
                                              on='club_name', how='left')

# Select and reorder columns
fact_participation = fact_participation[['athlete_id', 'participation_year', 'club_id', 'club_name',
                                         'competition_count', 'valid_rank_count', 
                                         'unique_sport_count', 'multi_sport_flag']]

# Add metadata
fact_participation['gold_created_timestamp'] = datetime.now()

print(f"Shape: {fact_participation.shape}")
print(f"Athlete-Year combinations: {len(fact_participation)}")
print(f"Multi-sport participation records: {fact_participation['multi_sport_flag'].sum()} ({fact_participation['multi_sport_flag'].sum()/len(fact_participation)*100:.2f}%)")
print(f"Sample:\n{fact_participation.head(5)}")

# Save to gold
output_path = os.path.join(GOLD_PATH, "fact_athlete_participation.csv")
fact_participation.to_csv(output_path, index=False)
print(f"\nSaved to: {output_path}")


CREATING: fact_athlete_participation
Shape: (27874, 9)
Athlete-Year combinations: 27874
Multi-sport participation records: 0 (0.00%)
Sample:
         athlete_id  participation_year  club_id club_name  competition_count  \
0  001O91NNW62RZP97                2015    550.0  DE LOVIE                  3   
1  001O91NNW62RZP97                2016    550.0  DE LOVIE                  3   
2  001O91NNW62RZP97                2017    550.0  DE LOVIE                  3   
3  001O91NNW62RZP97                2018    550.0  DE LOVIE                  3   
4  001O91NNW62RZP97                2019    550.0  DE LOVIE                  3   

   valid_rank_count  unique_sport_count  multi_sport_flag  \
0                 3                   1                 0   
1                 3                   1                 0   
2                 3                   1                 0   
3                 3                   1                 0   
4                 3                   1                 0   

    

In [18]:
# CREATE ATHLETE_CAREER
print("\n" + "="*60)
print("CREATING: athlete_career")
print("="*60)

# Aggregate across athlete career
athlete_career = results_silver.groupby('Code').agg({
    'year': ['min', 'max', 'nunique'],
    'rank_numeric': 'count',  # Total competitions with rank
    'score_numeric': ['count', 'mean', 'max', 'min'],
    'is_disqualified': 'sum',
    'sport_clean': 'nunique'
}).reset_index()

# Flatten column names
athlete_career.columns = ['athlete_id', 'first_year', 'last_year', 'years_active',
                          'total_competitions', 'competitions_with_score', 'avg_score',
                          'best_score', 'worst_score', 'disqualification_count', 'unique_sports']

# Calculate additional metrics
athlete_career['career_span_years'] = athlete_career['last_year'] - athlete_career['first_year'] + 1

# Join with dim_athlete to get athlete details
athlete_career = athlete_career.merge(dim_athlete[['athlete_id', 'person_type', 'gender', 'Age']], 
                                     on='athlete_id', how='left')

# Find favorite sport (most participated)
favorite_sport = results_silver.groupby(['Code', 'sport_clean']).size().reset_index(name='count')
favorite_sport = favorite_sport.loc[favorite_sport.groupby('Code')['count'].idxmax()]
favorite_sport = favorite_sport.rename(columns={'Code': 'athlete_id', 'sport_clean': 'favorite_sport'})

athlete_career = athlete_career.merge(favorite_sport[['athlete_id', 'favorite_sport']], 
                                     on='athlete_id', how='left')

# Add metadata
athlete_career['gold_created_timestamp'] = datetime.now()

print(f"Shape: {athlete_career.shape}")
print(f"Unique athletes in career table: {len(athlete_career)}")
print(f"Avg career span: {athlete_career['career_span_years'].mean():.2f} years")
print(f"Sample:\n{athlete_career.head(3)}")

# Save to gold
output_path = os.path.join(GOLD_PATH, "athlete_career.csv")
athlete_career.to_csv(output_path, index=False)
print(f"\nSaved to: {output_path}")


CREATING: athlete_career
Shape: (7607, 17)
Unique athletes in career table: 7607
Avg career span: 4.76 years
Sample:
         athlete_id  first_year  last_year  years_active  total_competitions  \
0  001O91NNW62RZP97        2015       2025             9                  22   
1  004SDDHB7VGAFGXR        2023       2024             2                   3   
2  00BKM94J4INLRQAF        2015       2025             9                  25   

   competitions_with_score  avg_score  best_score  worst_score  \
0                       24   3.125000        19.0          0.0   
1                        3   0.333333         1.0          0.0   
2                       39   6.794872        32.0          0.0   

   disqualification_count  unique_sports  career_span_years person_type  \
0                       2              1                 11     athlete   
1                       0              1                  2     athlete   
2                       2              3                 11     athlete

In [ ]:
# CREATE CLUB_METRICS 
print("\n" + "="*60)
print("CREATING: club_metrics")
print("="*60)

club_metrics = results_silver.groupby('Club').agg({
    'Code': 'nunique',  # Unique athletes
    'rank_numeric': 'count',  # Total competitions
    'score_numeric': ['mean', 'max'],
    'is_disqualified': 'sum',
    'year': 'nunique'
}).reset_index()


club_metrics.columns = ['club_name', 'unique_athletes', 'total_competitions',
                        'avg_score', 'best_score', 'disqualification_count', 'years_participated']

club_metrics = club_metrics.merge(dim_club[['club_id', 'club_name', 'region_id', 'region_name',
                                            'total_participations', 'certified_athlete_count']], 
                                 on='club_name', how='left')

# Reorder columns
club_metrics = club_metrics[['club_id', 'club_name', 'region_id', 'region_name',
                             'unique_athletes', 'certified_athlete_count', 'total_competitions',
                             'years_participated', 'avg_score', 'best_score',
                             'disqualification_count', 'total_participations']]


club_metrics['gold_created_timestamp'] = datetime.now()

print(f"Shape: {club_metrics.shape}")
print(f"Total clubs: {len(club_metrics)}")
print(f"Avg athletes per club: {club_metrics['unique_athletes'].mean():.2f}")
print(f"Sample:\n{club_metrics.head(5)}")

# Save to gold
output_path = os.path.join(GOLD_PATH, "club_metrics.csv")
club_metrics.to_csv(output_path, index=False)
print(f"\nSaved to: {output_path}")


CREATING: club_metrics
Shape: (497, 13)
Total clubs: 497
Avg athletes per club: 17.97
Sample:
   club_id          club_name  region_id      region_name  unique_athletes  \
0    693.0         'T ATELJEE       14.0        Antwerpen                4   
1    722.0  'T HAVERENHOF VZW       13.0   Vlaams Brabant                5   
2    675.0          'T PERDJE       14.0        Antwerpen               25   
3      NaN            'T VEER        NaN              NaN                4   
4    699.0       'T VELDZICHT       15.0  West-Vlaanderen               38   

   certified_athlete_count  total_competitions  years_participated  \
0                     11.0                  42                   5   
1                      6.0                  29                   4   
2                     18.0                 172                   8   
3                      NaN                  11                   4   
4                     57.0                 116                   9   

    avg_score  

In [ ]:
# CREATE ANNUAL_PERFORMANCE 
print("\n" + "="*60)
print("CREATING: annual_performance")
print("="*60)

annual_perf = results_silver.groupby(['year', 'sport_clean']).agg({
    'Code': 'nunique',  # Unique athletes
    'rank_numeric': 'count',  # Total results
    'score_numeric': ['count', 'mean', 'max', 'min'],
    'is_disqualified': 'sum',
    'Gender': lambda x: (x == 'M').sum()  # Count males
}).reset_index()

# Flatten column names
annual_perf.columns = ['year', 'sport_name', 'unique_athletes', 'total_results',
                       'results_with_score', 'avg_score', 'best_score', 'worst_score',
                       'disqualification_count', 'male_count']




annual_perf = annual_perf.merge(dim_sport[['sport_id', 'sport_name']], 
                               on='sport_name', how='left')

# Reorder columns
annual_perf = annual_perf[['sport_id', 'sport_name', 'year', 'unique_athletes', 'total_results',
                           'results_with_score', 'avg_score', 'best_score', 'worst_score',
                           'male_count', 'disqualification_count']]

# Add metadata
annual_perf['gold_created_timestamp'] = datetime.now()

print(f"Shape: {annual_perf.shape}")
print(f"Sport-Year combinations: {len(annual_perf)}")
print(f"Years covered: {sorted(annual_perf['year'].unique())}")
print(f"Sample:\n{annual_perf.head(5)}")

# Save to gold
output_path = os.path.join(GOLD_PATH, "annual_performance.csv")
annual_perf.to_csv(output_path, index=False)
print(f"\nSaved to: {output_path}")

# Save to gold


CREATING: annual_performance
Shape: (175, 12)
Sport-Year combinations: 175
Years covered: [np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
Sample:
   sport_id                   sport_name  year  unique_athletes  \
0        11  Adapted Physical Activities  2015              105   
1         6                     Aquatics  2015              530   
2         1                    Athletics  2015              781   
3         7                    Badminton  2015              146   
4        17                   Basketball  2015              168   

   total_results  results_with_score  avg_score  best_score  worst_score  \
0              0                 702  86.796296       940.0          0.0   
1           1087                1269   1.052009         8.0          0.0   
2           1731                1954   3.099795        38.0          0.0   
3            736                 642   6.056075  